In [1]:
import os
import shutil
import random
import uuid
from pathlib import Path

In [2]:
BASE_PATH = Path("../data")

DATASETS = {
    "chest": BASE_PATH / "chest_xray_multi" / "train",
    "bone": BASE_PATH / "bone_xray" / "train",
    "dental": BASE_PATH / "dental_xray" / "train",
    "knee": BASE_PATH / "knee_xray" / "train"
}

OUTPUT_PATH = BASE_PATH / "type_classifier"
MODEL_PATH = Path("../backend/saved_models/type_classifier.keras")
CLASS_MAP_PATH = Path("../backend/saved_models/type_classes.json")

In [3]:
if OUTPUT_PATH.exists():
    shutil.rmtree(OUTPUT_PATH)

for split in ["train", "val", "test"]:
    for label in DATASETS.keys():
        os.makedirs(OUTPUT_PATH / split / label, exist_ok=True)

print("✅ Clean directory created")

✅ Clean directory created


In [4]:
def get_all_images(folder):
    images = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(root, f))
    return images

In [5]:
SAMPLES_PER_CLASS = {
    "chest": 500,
    "bone": 500,
    "dental": 250,
    "knee": 250
}

SPLIT_RATIO = (0.7, 0.15, 0.15)

for label, folder in DATASETS.items():
    all_images = get_all_images(folder)
    random.shuffle(all_images)

    selected = all_images[:SAMPLES_PER_CLASS[label]]

    n = len(selected)
    train_end = int(n * SPLIT_RATIO[0])
    val_end = int(n * (SPLIT_RATIO[0] + SPLIT_RATIO[1]))

    splits = {
        "train": selected[:train_end],
        "val": selected[train_end:val_end],
        "test": selected[val_end:]
    }

    for split, imgs in splits.items():
        for img in imgs:
            # ✅ UNIQUE NAME (CRITICAL FIX)
            unique_name = f"{uuid.uuid4().hex}_{os.path.basename(img)}"
            dst = OUTPUT_PATH / split / label / unique_name
            shutil.copy(img, dst)

print("✅ Dataset created safely (no overwrite)")

✅ Dataset created safely (no overwrite)


In [6]:
from collections import defaultdict

counts = defaultdict(dict)

for split in ["train", "val", "test"]:
    for label in DATASETS.keys():
        path = OUTPUT_PATH / split / label
        counts[split][label] = len(os.listdir(path))

print("\n📊 Dataset Distribution:")
for split, labels in counts.items():
    print(f"\n{split.upper()}")
    for label, count in labels.items():
        print(f"{label}: {count}")


📊 Dataset Distribution:

TRAIN
chest: 350
bone: 350
dental: 175
knee: 175

VAL
chest: 75
bone: 75
dental: 37
knee: 37

TEST
chest: 75
bone: 75
dental: 38
knee: 38


In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = OUTPUT_PATH / "train"
VAL_DIR   = OUTPUT_PATH / "val"
TEST_DIR  = OUTPUT_PATH / "test"

In [9]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 1050 images belonging to 4 classes.
Found 224 images belonging to 4 classes.


In [10]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        save_best_only=True
    )
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - accuracy: 0.5971 - loss: 1.0011 - val_accuracy: 0.9554 - val_loss: 0.3894
Epoch 2/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9105 - loss: 0.3283 - val_accuracy: 0.9643 - val_loss: 0.2017
Epoch 3/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9571 - loss: 0.1717 - val_accuracy: 0.9732 - val_loss: 0.1459
Epoch 4/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9752 - loss: 0.1144 - val_accuracy: 0.9732 - val_loss: 0.1148
Epoch 5/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.9676 - loss: 0.1037 - val_accuracy: 0.9732 - val_loss: 0.1038
Epoch 6/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 39s 1s/step - accuracy: 0.9829 - loss: 0.0793 - val_accuracy: 0.9732 - val_loss: 0.0996
Epoch 7/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9857 - loss: 0.0662 - val_accuracy: 0.9777 - val_loss: 0.0859
Epoch 8/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.9867 - loss: 0.0583 - val_accuracy: 0.9777 - val_loss:

In [ ]:
test_gen = val_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

loss, acc = model.evaluate(test_gen)
print("✅ Test Accuracy:", acc)

Found 226 images belonging to 4 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.9867 - loss: 0.0414
✅ Test Accuracy: 0.9867256879806519


In [13]:
import json

with open(CLASS_MAP_PATH, "w") as f:
    json.dump(train_gen.class_indices, f)

print("✅ Class mapping saved!")

✅ Class mapping saved!
